# TPC_RP: Algorithm Modification

# Step 1: Set up

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
import numpy as np

In [ ]:
# Access to GPU compute is advised for training -> Check if GPU is enabled
print(torch.cuda.is_available())  # should print True
print(torch.cuda.get_device_name(0))  # should print the GPU name
print(f"Memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Define modification

In [ ]:
def query_selection_with_temperature(embeddings, cluster_assignments, labelled_indices, B, iteration, tau_0=0.01, gamma=1.5):

    # Steps 1 & 2 unchanged
    covered = set(cluster_assignments[i] for i in labelled_indices)
    K = cluster_assignments.max() + 1
    uncovered = [c for c in range(K) if c not in covered]

    cluster_sizes = {c: np.sum(cluster_assignments == c) for c in uncovered}
    valid_uncovered = [c for c in uncovered if cluster_sizes[c] >= 5]
    largest_B = sorted(valid_uncovered, key=lambda c: -cluster_sizes[c])[:B]

    # Step 3 modified — soft selection within each cluster
    tau = tau_0 * (gamma ** iteration)

    queries = []
    for cluster_id in largest_B:
        cluster_indices = np.where(cluster_assignments == cluster_id)[0]
        cluster_embs = embeddings[cluster_indices]

        K_neighbours = min(20, len(cluster_embs))
        typicality_scores = Typicality(cluster_embs, K=K_neighbours, normalised=True)

        if tau < 1e-6:  # negligible temperature -> behave like original argmax
            best_local = np.argmax(typicality_scores)
        else:
            logits = typicality_scores / tau
            logits -= logits.max()  # subtract max for numerical stability before exp
            probs = np.exp(logits)
            probs /= probs.sum()
            best_local = np.random.choice(len(cluster_indices), p=probs)

        queries.append(cluster_indices[best_local])

    return queries

# Step 3: Prepare improved TPC_RP (most of the algorithm remains unchanged)

In [ ]:
def kmeans_cluster(embeddings, K):
    """
    embeddings: numpy array of shape (N, 512)
    K: number of clusters = min(|L_prev| + B, max_clusters)
    returns: cluster assignment (labels -> indices) for each of the N points
    """
    # Paper uses KMeans for K<=50, MiniBatchKMeans otherwise (Appendix F.1)
    if K <= 50:
        kmeans = KMeans(n_clusters=K, random_state=42)
    else:
        kmeans = MiniBatchKMeans(n_clusters=K, random_state=42)

    cluster_assignments = kmeans.fit_predict(embeddings)  # (N,) array
    return cluster_assignments

In [ ]:
def Typicality(cluster_embeddings, K=20, normalised=False):
    """
    cluster_embeddings: (M, 512) array of embeddings for points in ONE cluster
    K: number of nearest neighbours, paper uses K=20
    normalised: whether typicality scores should be normalised to [0, 1]
    returns: (M,) array of typicality scores, one per point
    """

    k = min(K + 1, len(cluster_embeddings))
    neighbours = NearestNeighbors(n_neighbors=k).fit(cluster_embeddings)
    distances, _ = neighbours.kneighbors(cluster_embeddings)

    # distances[:, 0] is always 0 (self), so skip it
    avg_distances = distances[:, 1:].mean(axis=1)  # (M,)

    typicality = 1.0 / (avg_distances + 1e-8) # prevent division by zero error

    if normalised:
        typicality = typicality - typicality.min()
        typicality = typicality / (typicality.max() + 1e-8)

    return typicality

In [ ]:
def query_selection(embeddings, cluster_assignments, labelled_indices, B):
    """
    embeddings:          (N, 512) numpy array - all 50k embeddings
    cluster_assignments: (N,) numpy array - cluster id for each image
    labelled_indices:    list of ints - indices of images already labelled
    B:                   int - how many new points to query this round (budget)

    returns: list of B indices to query
    """
    # 1. Find uncovered clusters
    covered = set(cluster_assignments[i] for i in labelled_indices)
    K = cluster_assignments.max() + 1  # total number of clusters
    uncovered = [c for c in range(K) if c not in covered]

    # 2. Sort uncovered by cluster size, take B largest (ignore clusters with fewer than 5 points)
    cluster_sizes = {c: np.sum(cluster_assignments == c) for c in uncovered}
    valid_uncovered = [c for c in uncovered if cluster_sizes[c] >= 5]
    largest_B = sorted(valid_uncovered, key=lambda c: -cluster_sizes[c])[:B]

    # 3. For each selected cluster, pick most typical point
    queries = []
    for cluster_id in largest_B:
        cluster_indices = np.where(cluster_assignments == cluster_id)[0] # indices of images in this cluster
        cluster_embs = embeddings[cluster_indices] # embeddings of the images in this cluster

        K_neighbours = min(20, len(cluster_embs))
        typicality_scores = Typicality(cluster_embs, K=K_neighbours) # array of all typicality scores of the embeddings in this cluster
        best_local = np.argmax(typicality_scores) # best typicality scores (indexed locally i.e. indices of typicality scores == cluster_indices == cluster_embs)
        best_global = cluster_indices[best_local] # get global/actual index of image
        queries.append(best_global)

    return queries

# Modified TCP_RP Final Implementation

# Step 1: Set up

In [ ]:
## SimCLR model def
# First load the ResNet-18 classifier
resnet18 = torchvision.models.resnet18(weights=None, progress=True)

# Reduce kernel size and stride as CIFAR10 images are very small
resnet18.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

# Remove max pooling layer for same reason
resnet18.maxpool = nn.Identity()

# Remove final classification layer (as embeddings are in penultimate layer)
resnet18.fc = nn.Identity()

# Define projection head (only used during training)
class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(512, 512)
        self.bn = nn.BatchNorm1d(512)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(512, 128)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.fc2(x)
        return F.normalize(x, dim=1)

# Define SimCLR model (a wrapper of the ResNet-18 and projection head)
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = resnet18
        self.projection = ProjectionHead()

    def forward(self, x1, x2):
        h1 = self.encoder(x1)  # (batch_size, 512)
        h2 = self.encoder(x2)  # (batch_size, 512)

        z1 = self.projection(h1)  # (batch_size, 128)
        z2 = self.projection(h2)  # (batch_size, 128)

        return z1, z2

    # Use after training for embedding extraction
    def get_embedding(self, x):
        return self.encoder(x)

In [ ]:
# Load saved model
checkpoint_path_simclr = '/content/drive/MyDrive/5CCSAMLF_CW2/models/simclr_latest.pth'
checkpoint_simclr = torch.load(checkpoint_path_simclr)

simclr = SimCLR().to(device)
simclr.load_state_dict(checkpoint_simclr['model_state_dict'])

In [ ]:
# Load saved train embeddings + labels
checkpoint_path_train = '/content/drive/MyDrive/5CCSAMLF_CW2/models/train_embeddings.pth'
checkpoint_train = torch.load(checkpoint_path_train)

train_embeddings = checkpoint_train['embeddings']
train_labels = checkpoint_train['labels']

In [ ]:
# Load saved test embeddings + labels
checkpoint_path_test = '/content/drive/MyDrive/5CCSAMLF_CW2/models/test_embeddings.pth'
checkpoint_test = torch.load(checkpoint_path_test)

test_embeddings = checkpoint_test['embeddings']
test_labels = checkpoint_test['labels']

In [ ]:
# Define train/evaluation loop to evaluate TCP_RP AL strategy
def train_and_evaluate(labelled_indices, num_epochs=200):
    """
    Train a linear classifier on the labelled embeddings, evaluate on CIFAR-10 test set.
    Returns (acc, linear) so the model can be reused for query scoring.
    """
    X_train = train_embeddings[labelled_indices].to(device)
    y_train = train_labels[labelled_indices].to(device)
    X_test = test_embeddings.to(device)
    y_test = test_labels.to(device)

    linear = nn.Linear(512, 10).to(device)
    optimiser = torch.optim.SGD(linear.parameters(), lr=2.5,
                                momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=num_epochs)
    criterion = nn.CrossEntropyLoss()

    linear.train()
    for epoch in range(num_epochs):
        optimiser.zero_grad()
        outputs = linear(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimiser.step()
        scheduler.step()

    linear.eval()
    with torch.no_grad():
        test_outputs = linear(X_test)
        preds = test_outputs.argmax(dim=1)
        acc = (preds == y_test).float().mean().item() * 100
    return acc, linear

# Step 2: Implementation (AL loop)

In [ ]:
'''
For each iteration, new labels are selected to be queried, then a linear classifier is
trained on those labels and evaluated.
'''

N_ROUNDS = 10
N_ITERATIONS = 6
B = 10

accuracies_mat_typiclust = []
accuracies_mat_random = []
accuracies_mat_modified = []

train_embeddings_np = train_embeddings.numpy()

for round_idx in range(N_ROUNDS):
    print(f"\n--- Round {round_idx+1}/{N_ROUNDS} ---")

    labelled_indices_typi = []
    labelled_indices_rand = []
    labelled_indices_modified = []

    round_accs_typi = []
    round_accs_rand = []
    round_accs_modified = []

    for iteration in range(N_ITERATIONS):

        K = min(len(labelled_indices_typi) + B, 500)
        cluster_assignments = kmeans_cluster(train_embeddings_np, K)

        # ---- TypiClust queries ----
        new_queries_typi = query_selection(train_embeddings_np, cluster_assignments,labelled_indices_typi, B)
        labelled_indices_typi.extend([int(q) for q in new_queries_typi])
        acc_typi, _ = train_and_evaluate(labelled_indices_typi)

        # ---- Random queries ----
        available = list(set(range(len(train_labels))) - set(labelled_indices_rand))
        labelled_indices_rand.extend(
            np.random.choice(available, B, replace=False).tolist()
        )
        acc_rand, _ = train_and_evaluate(labelled_indices_rand)

        # ---- Modified TypiClust queries ----
        new_queries_modified = query_selection_with_temperature(train_embeddings_np,
                                                                cluster_assignments,
                                                                labelled_indices_modified,
                                                                B,
                                                                iteration=iteration,
                                                                tau_0=0.1, gamma=1.5)
        labelled_indices_modified.extend(new_queries_modified)
        acc_modified, _ = train_and_evaluate(labelled_indices_modified)

        # ---- Append results ----

        round_accs_typi.append(acc_typi)
        round_accs_rand.append(acc_rand)
        round_accs_modified.append(acc_modified)

        print(f"Iteration {iteration+1} | Budget {len(labelled_indices_typi)} | "
              f"TPC_RP: {acc_typi:.1f}% | Rand: {acc_rand:.1f}% | "
              f"Modified: {acc_modified:.1f}%")


    accuracies_mat_typiclust.append(round_accs_typi)
    accuracies_mat_random.append(round_accs_rand)
    accuracies_mat_modified.append(round_accs_modified)

In [ ]:
## Ablation
# Compare modified algorithm with different values of tau_0 and gamma (low, moderate and aggressive scheduling)

N_ROUNDS = 10
N_ITERATIONS = 6
B = 10

accuracies_mat_low = []
accuracies_mat_med = []
accuracies_mat_high = []
accuracies_mat_benchmark = []

train_embeddings_np = train_embeddings.numpy()

for round_idx in range(N_ROUNDS):
    print(f"\n--- Round {round_idx+1}/{N_ROUNDS} ---")

    labelled_indices_low = []
    labelled_indices_med = []
    labelled_indices_high = []
    labelled_indices_benchmark = []

    round_accs_low = []
    round_accs_med = []
    round_accs_high = []
    round_accs_benchmark = []

    for iteration in range(N_ITERATIONS):

        K = min(len(labelled_indices_low) + B, 500) # They all grow at the same rate

        # The same cluster assignments will be used across all variants for fair comparison
        cluster_assignments = kmeans_cluster(train_embeddings_np, K)

        # ---- Benchmark TypiClust queries ----
        new_queries_benchmark = query_selection(train_embeddings_np, cluster_assignments,labelled_indices_benchmark, B)
        labelled_indices_benchmark.extend([int(q) for q in new_queries_benchmark])
        acc_benchmark, _ = train_and_evaluate(labelled_indices_benchmark)

        # ---- Low intensity ----
        new_queries_low = query_selection_with_temperature(train_embeddings_np,
                                                                cluster_assignments,
                                                                labelled_indices_low,
                                                                B,
                                                                iteration=iteration,
                                                                tau_0=1.0, gamma=1.2)
        labelled_indices_low.extend(new_queries_low)
        acc_low, _ = train_and_evaluate(labelled_indices_low)

        # ---- Moderate intensity ----
        new_queries_med = query_selection_with_temperature(train_embeddings_np,
                                                                cluster_assignments,
                                                                labelled_indices_med,
                                                                B,
                                                                iteration=iteration,
                                                                tau_0=1.0, gamma=1.5)
        labelled_indices_med.extend(new_queries_med)
        acc_med, _ = train_and_evaluate(labelled_indices_med)

        # ---- High intensity ----
        new_queries_high = query_selection_with_temperature(train_embeddings_np,
                                                                cluster_assignments,
                                                                labelled_indices_high,
                                                                B,
                                                                iteration=iteration,
                                                                tau_0=1.0, gamma=3)
        labelled_indices_high.extend(new_queries_high)
        acc_high, _ = train_and_evaluate(labelled_indices_high)


        # ---- Append results ----

        round_accs_low.append(acc_low)
        round_accs_med.append(acc_med)
        round_accs_high.append(acc_high)
        round_accs_benchmark.append(acc_benchmark)

        print(f"Iteration {iteration+1} | Budget {len(labelled_indices_low)} | "
              f"Low: {acc_low:.1f}% | Moderate: {acc_med:.1f}% | "
              f"High: {acc_high:.1f}% | TPC_RP (benchmark): {acc_benchmark:.1f}%")


    accuracies_mat_high.append(round_accs_high)
    accuracies_mat_med.append(round_accs_med)
    accuracies_mat_low.append(round_accs_low)
    accuracies_mat_benchmark.append(round_accs_benchmark)

# Step 3: Results and visualisations

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def fmt(means, stds):
    return [f"{m:.1f} ± {s:.1f}" for m, s in zip(means, stds)]

Visualisations for first experiment

In [ ]:
# Compute mean and std for plots

accuracies_mat_benchmark = np.array(accuracies_mat_typiclust)
accuracies_mat_random = np.array(accuracies_mat_random)
accuracies_mat_modified = np.array(accuracies_mat_modified)

mean_typiclust = accuracies_mat_typiclust.mean(axis=0)  # (6,) - mean per iteration
mean_random = accuracies_mat_random.mean(axis=0)
mean_modified = accuracies_mat_modified.mean(axis=0)

std_typiclust = accuracies_mat_typiclust.std(axis=0)   # for error bars in plots
std_random = accuracies_mat_random.std(axis=0)
std_modified = accuracies_mat_modified.std(axis=0)

budgets = [B * (i+1) for i in range(N_ITERATIONS)]  # [10, 20, 30, 40, 50, 60]

In [ ]:
# Form results table

df = pd.DataFrame({
    'Budget': budgets,
    'TPC_RP': fmt(mean_typiclust, (std_typiclust / np.sqrt(N_ROUNDS))), # use stderr
    'Random': fmt(mean_random, (std_random / np.sqrt(N_ROUNDS))),
    'Modified (normalisation)': fmt(mean_modified, (std_modified / np.sqrt(N_ROUNDS))),
})

df = df.set_index('Budget')

print(df.to_string())
print(df.to_latex())

In [ ]:
# Line plot
plt.figure(figsize=(8, 6))

plt.plot(budgets, mean_typiclust, label='TPC_RP')
plt.fill_between(budgets, mean_typiclust - (std_typiclust / np.sqrt(N_ROUNDS)), mean_typiclust + (std_typiclust / np.sqrt(N_ROUNDS)), alpha=0.2)

plt.plot(budgets, mean_random, label='Random')
plt.fill_between(budgets, mean_random - (std_random / np.sqrt(N_ROUNDS)), mean_random + (std_random / np.sqrt(N_ROUNDS)), alpha=0.2)

plt.plot(budgets, mean_modified, label='Modified')
plt.fill_between(budgets, mean_modified - (std_modified / np.sqrt(N_ROUNDS)), mean_modified + (std_modified / np.sqrt(N_ROUNDS)), alpha=0.2)

plt.ylabel('Mean Test Accuracy (%)')
plt.xlabel('Cumulative Budget')
plt.title(
    'Fully Supervised with Self-Supervised Embeddings – CIFAR-10\n'
    '10 independent runs, shaded region = ±1 SE\n'
    'τ_0 = 1',
    )
plt.grid(True, alpha=0.3)
plt.legend()

plt.savefig('improved_results_with_se.png')
plt.savefig('improved_results_with_se.svg')

plt.show()

Visualisations for second experiment

In [ ]:
accuracies_mat_low = np.array(accuracies_mat_low)
accuracies_mat_med = np.array(accuracies_mat_med)
accuracies_mat_high = np.array(accuracies_mat_high)
accuracies_mat_benchmark = np.array(accuracies_mat_benchmark)

mean_low = accuracies_mat_low.mean(axis=0)
mean_med = accuracies_mat_med.mean(axis=0)
mean_high = accuracies_mat_high.mean(axis=0)
mean_benchmark = accuracies_mat_benchmark.mean(axis=0)

std_low = accuracies_mat_low.std(axis=0)
std_med = accuracies_mat_med.std(axis=0)
std_high = accuracies_mat_high.std(axis=0)
std_benchmark = accuracies_mat_benchmark.std(axis=0)

budgets = [B * (i+1) for i in range(N_ITERATIONS)]  # [10, 20, 30, 40, 50, 60]

In [ ]:
# Form results table

df = pd.DataFrame({
    'Budget': budgets,
    'TPC_RP (benchmark)': fmt(mean_benchmark, (std_benchmark / np.sqrt(N_ROUNDS))),
    'Low': fmt(mean_low, (std_low / np.sqrt(N_ROUNDS))),
    'Moderate': fmt(mean_med, (std_med / np.sqrt(N_ROUNDS))),
    'High': fmt(mean_high, (std_high / np.sqrt(N_ROUNDS))),
})

df = df.set_index('Budget')

print(df.to_string())
print(df.to_latex())

In [ ]:
# Line plot
plt.figure(figsize=(8, 6))

plt.plot(budgets, mean_benchmark, label='TPC_RP (benchmark)')
plt.fill_between(budgets, mean_benchmark - (std_benchmark / np.sqrt(N_ROUNDS)), mean_benchmark + (std_typiclust / np.sqrt(N_ROUNDS)), alpha=0.2)

plt.plot(budgets, mean_low, label='Low (τ₀ = 0.1, γ = 1.2)')
plt.fill_between(budgets, mean_low - (std_low / np.sqrt(N_ROUNDS)), mean_low + (std_low / np.sqrt(N_ROUNDS)), alpha=0.2)

plt.plot(budgets, mean_med, label='Moderate (τ₀ = 0.1, γ = 1.5)')
plt.fill_between(budgets, mean_med - (std_med / np.sqrt(N_ROUNDS)), mean_med + (std_med / np.sqrt(N_ROUNDS)), alpha=0.2)

plt.plot(budgets, mean_high, label='High (τ₀ = 0.1, γ = 3)')
plt.fill_between(budgets, mean_high - (std_high / np.sqrt(N_ROUNDS)), mean_high + (std_high / np.sqrt(N_ROUNDS)), alpha=0.2)

plt.ylabel('Mean Test Accuracy (%)')
plt.xlabel('Cumulative Budget')
plt.title(
    'Fully Supervised with Self-Supervised Embeddings – CIFAR-10\n'
    '10 independent runs, shaded region = ±1 SE',
    )
plt.grid(True, alpha=0.3)
plt.legend()

plt.savefig('improved_results_ablation_SMALL.png')

plt.show()